# Mock de MLflow: experimento y modelo sin entrenar

Este notebook **no entrena ningún modelo**. Sirve para comprobar la conexión con MLflow:

1. Se conecta al servidor de tracking.
2. Registra un experimento con un run y métricas fijas.
3. Registra un modelo falso en el Model Registry y lo carga de nuevo.

Requisito: MLflow levantado (ver `services/README.md`), en `http://localhost:5000`.

## 1. Conexión con MLflow

In [ ]:
import os

import mlflow
from mlflow import MlflowClient

TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI", "http://localhost:5000")
mlflow.set_tracking_uri(TRACKING_URI)
client = MlflowClient()

print("Tracking URI:", mlflow.get_tracking_uri())
print("Experimentos existentes:", [e.name for e in client.search_experiments()])

## 2. Experimento y run con métricas fijas

In [ ]:
EXPERIMENTO = "mock-churn-v0"
mlflow.set_experiment(EXPERIMENTO)

with mlflow.start_run(run_name="mock-run") as run:
    mlflow.log_params({"modelo": "mock", "entrenado": False})
    mlflow.log_metrics({"auc": 0.75, "recall": 0.60})
    run_id = run.info.run_id

print("run_id:", run_id)

## 3. Modelo falso

`ModeloMock` no aprende nada: devuelve siempre la misma probabilidad de churn. Basta para probar el registro y la carga desde MLflow.

In [ ]:
import numpy as np
from mlflow.pyfunc import PythonModel


class ModeloMock(PythonModel):
    def __init__(self, probabilidad=0.3):
        self.probabilidad = probabilidad

    def predict(self, context, model_input, params=None):
        return np.full(len(model_input), self.probabilidad)

## 4. Registrar el modelo en el Model Registry

In [ ]:
NOMBRE_MODELO = "churn-mock"

with mlflow.start_run(run_id=run_id):
    info = mlflow.pyfunc.log_model(
        name="modelo",
        python_model=ModeloMock(),
        registered_model_name=NOMBRE_MODELO,
    )

print("model_uri:", info.model_uri)

## 5. Comprobar el modelo registrado

In [ ]:
import pandas as pd

versiones = client.search_model_versions(f"name='{NOMBRE_MODELO}'")
ultima = max(versiones, key=lambda v: int(v.version))
print("Versión:", ultima.version, "| estado:", ultima.status)

modelo = mlflow.pyfunc.load_model(f"models:/{NOMBRE_MODELO}/{ultima.version}")
print("Predicciones:", modelo.predict(pd.DataFrame({"x": [1, 2, 3]})))

## 6. Dónde verlo

En la UI de MLflow (http://localhost:5000):

- Experimento `mock-churn`, con el run `mock-run` y sus métricas.
- Modelo `churn-mock` en el Model Registry, con la versión creada aquí.

## 7. Explorar MLflow a fondo

Hasta aquí el notebook registra y carga un modelo. Esta sección muestra cómo consultar lo que hay en el servidor, añadir tags, leer métricas y encontrar los archivos del modelo.

Todo se hace con `MlflowClient`, que habla con el servidor por HTTP. Conceptos clave:

| Concepto | Qué es |
|---|---|
| **Experimento** | Agrupa runs relacionados (aquí, `mock-churn`). |
| **Run** | Una ejecución. Guarda parámetros, métricas, tags y artefactos. |
| **Modelo registrado** | Nombre estable del modelo en el Model Registry (`churn-mock`). |
| **Versión** | Cada registro del modelo. Apunta al modelo logueado en `source`. |
| **Tag** | Par clave-valor que se puede poner a un run, a un modelo registrado o a una versión. |
| **Alias** | Nombre móvil que apunta a una versión (por ejemplo `champion`). En MLflow 3 se usa en lugar de los *stages*, que aquí aparecen vacíos. |

### 7.1 Consultar versiones y el modelo registrado

`search_model_versions` acepta un filtro con sintaxis parecida a SQL y permite ordenar. `get_model_version` devuelve una versión concreta.

In [ ]:
versiones = client.search_model_versions(
    f"name='{NOMBRE_MODELO}'", order_by=["version_number DESC"]
)
for v in versiones:
    print(f"v{v.version} | run={v.run_id} | estado={v.status} | source={v.source}")

v2 = client.get_model_version(NOMBRE_MODELO, "2")
print("\nv2 tags:", v2.tags, "| descripción:", v2.description)

modelo_registrado = client.get_registered_model(NOMBRE_MODELO)
print("\nCreado:", pd.to_datetime(modelo_registrado.creation_timestamp, unit="ms"))
print("Tags:", modelo_registrado.tags)
print("Aliases:", modelo_registrado.aliases)

### 7.2 Tags y aliases

Hay tres niveles de tags: el **run**, el **modelo registrado** (común a todas sus versiones) y la **versión**. Un `set_*` con una clave que ya existe la sobrescribe.

Los aliases apuntan a una versión por nombre. Con `models:/<nombre>@<alias>` se carga siempre la versión a la que apunta el alias, sin conocer el número.

In [ ]:
# Tag en el modelo registrado (se ve en todas sus versiones)
client.set_registered_model_tag(NOMBRE_MODELO, "dominio", "churn")

# Tag en una versión concreta
client.set_model_version_tag(NOMBRE_MODELO, ultima.version, "validado", "no")

# Tag en un run
client.set_tag(run_id, "revisado_por", "equipo-mlops")

print("Modelo:", client.get_registered_model(NOMBRE_MODELO).tags)
print("Versión:", client.get_model_version(NOMBRE_MODELO, ultima.version).tags)

# Alias: 'champion' apunta a la última versión
client.set_registered_model_alias(NOMBRE_MODELO, "champion", ultima.version)
campeon = client.get_model_version_by_alias(NOMBRE_MODELO, "champion")
print("\nchampion -> versión", campeon.version)

modelo_por_alias = mlflow.pyfunc.load_model(f"models:/{NOMBRE_MODELO}@champion")
print("Predicciones:", modelo_por_alias.predict(pd.DataFrame({"x": [1, 2, 3]})))

### 7.3 Obtener métricas

Un run guarda sus métricas finales en `run.data.metrics`. Si una métrica se registra varias veces (por ejemplo, por época), `get_metric_history` devuelve todo el historial.

Para comparar runs, `mlflow.search_runs` devuelve un DataFrame con una fila por run.

In [ ]:
run = client.get_run(run_id)
print("Métricas finales:", run.data.metrics)
print("Parámetros:", run.data.params)

# Historial de una métrica (aquí solo hay un punto)
for m in client.get_metric_history(run_id, "auc"):
    print("paso", m.step, "| valor", m.value, "| timestamp", m.timestamp)

# Métricas del run del que sale la última versión del modelo
print(f"\nMétricas de la versión {ultima.version}:", client.get_run(ultima.run_id).data.metrics)

# Comparar todos los runs del experimento
runs = mlflow.search_runs(
    experiment_names=[EXPERIMENTO],
    order_by=["metrics.auc DESC"],
    output_format="pandas",
)
runs[["run_id", "tags.mlflow.runName", "metrics.auc", "metrics.recall"]]

### 7.4 Dónde se guardan los artefactos

Hay dos sitios distintos:

- **Metadatos** (experimentos, runs, params, métricas, tags, versiones): en una base de datos SQLite, `services/mlflow_store/mlflow.db`, según `--backend-store-uri` en `services/docker-compose.yml`.
- **Artefactos** (ficheros del modelo, gráficas, etc.): en `services/mlflow_store/artifacts/`, según `--artifacts-destination`. Dentro hay una carpeta por experimento, como `2/`. Ruta en Windows: `services\mlflow_store\artifacts\`.

El `artifact_uri` del run usa el esquema `mlflow-artifacts:/`. El cliente no accede al disco directamente: pide los ficheros al servidor, que los lee de esa carpeta. Por eso funciona igual desde otro equipo.

En MLflow 3, el modelo que se loguea con `log_model` se guarda como **modelo logueado**, separado del run. Por eso `list_artifacts` sobre el run puede aparecer vacío, y los ficheros del modelo están en `artifacts/<experimento>/models/<id>/artifacts/`.

> No borres ni muevas esas carpetas a mano: la base de datos seguiría apuntando a ficheros que ya no existen. Para borrar, usa la API o la UI.

In [ ]:
from pathlib import Path

print("artifact_uri del run:", client.get_run(run_id).info.artifact_uri)
print("source de la versión:", ultima.source)

# Descarga los ficheros del modelo a una carpeta temporal
descarga = mlflow.artifacts.download_artifacts(artifact_uri=f"models:/{NOMBRE_MODELO}/{ultima.version}")
print("\nDescargado en:", descarga)
for f in sorted(Path(descarga).rglob("*")):
    if f.is_file():
        print(" -", f.relative_to(descarga))

### 7.5 Limpieza (opcional)

Quita los tags y el alias creados en esta sección. Las versiones y los runs se conservan.

In [ ]:
client.delete_registered_model_alias(NOMBRE_MODELO, "champion")
client.delete_registered_model_tag(NOMBRE_MODELO, "dominio")
client.delete_model_version_tag(NOMBRE_MODELO, ultima.version, "validado")
client.delete_tag(run_id, "revisado_por")
print("Tags y alias eliminados")